# Arbiter fixture research artifact

This notebook inspects the versioned tables produced by `arbiter report`. It deliberately does not recompute production metrics. The tracked fixture is synthetic pipeline evidence, not an empirical sample of Kalshi behavior, realized profit, or atomic fillability. See [`docs/RESEARCH.md`](../docs/RESEARCH.md) for the cohort, censoring, and aggregation rules.

## Generate inputs

From the repository root, run:

```bash
.venv/bin/arbiter report --db tests/fixtures/research/arbiter.duckdb --output-dir build/fixture-report
```

The report command owns cohort deduplication, episode aggregation, market-hour exposure, and right-censor handling.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "arbiter").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the Arbiter repository.")


ROOT = find_repository_root(Path.cwd().resolve())
REPORT_DIR = ROOT / "build" / "fixture-report"
required = {
    "summary": REPORT_DIR / "summary.csv",
    "funnel": REPORT_DIR / "funnel.csv",
    "breakdowns": REPORT_DIR / "breakdowns.csv",
    "episodes": REPORT_DIR / "episodes.parquet",
}
missing = [str(path.relative_to(ROOT)) for path in required.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Generate the fixture report first; missing: " + ", ".join(missing))

summary = pd.read_csv(required["summary"], keep_default_na=False)
funnel = pd.read_csv(required["funnel"], keep_default_na=False)
breakdowns = pd.read_csv(required["breakdowns"], keep_default_na=False)
episodes = pd.read_parquet(required["episodes"])
print(f"Loaded {len(episodes)} unique fixture episode(s) from {REPORT_DIR.relative_to(ROOT)}")

## Fixed funnel and summary

Funnel rows count unique episodes. Repeated `UPDATED` observations and repeated replays of one recording are not independent observations. Undefined rates remain blank rather than being replaced with zero.

In [ ]:
display(funnel)
display(summary)

## Episode evidence and censoring

A closed duration is observed. A right-censored duration is only a lower bound. Likewise, a paper attempt with `insufficient_future_data` is unevaluable rather than a failed fill. The required episode columns below make those distinctions explicit.

In [ ]:
episode_columns = [
    "fact_id",
    "cohort_id",
    "run_id",
    "opportunity_id",
    "component_id",
    "opened_at",
    "terminal_status",
    "ended_at",
    "duration_seconds",
    "relation_types",
    "relation_sources",
    "categories",
    "settlement_bucket",
    "peak_stage",
    "peak_gross_edge",
    "peak_net_edge",
    "peak_capacity",
    "peak_net_guarantee",
    "midpoint_available",
    "reference_violation",
    "one_contract_gross_survived",
    "one_contract_fee_survived",
    "depth_executable",
    "paper_status",
    "paper_locked_profit",
]
missing_episode_columns = [column for column in episode_columns if column not in episodes.columns]
if missing_episode_columns:
    raise ValueError("Episode artifact is missing columns: " + ", ".join(missing_episode_columns))
display(episodes.loc[:, episode_columns])

## Breakdown tables

Relation and category rows may be multi-label, so their counts need not add to the overall episode count. Settlement buckets are mutually exclusive and additive. Associations in this synthetic fixture are not evidence of a venue-wide pattern.

In [ ]:
display(breakdowns)

## Interpretation boundary

The sum of episode guarantees is hypothetical, overlapping, and non-independent. It is not realized P&L: episodes may reuse capital or displayed liquidity, and multi-leg execution is not atomic. Answering the ten research questions requires a declared live collection cohort; this notebook only demonstrates a reproducible inspection path for the fixture.